# V2G gap study — IEEE-34, fleet-constrained multi-hub, degradation-aware

Extends Wang, Jacob, Kaushik & Zhang, *"Reinforcement Learning for Vehicle-to-Grid Voltage
Regulation: Single-Hub to Multi-Hub Coordination with Battery-Aware Constraints"*
([arXiv:2603.07237](https://arxiv.org/abs/2603.07237)) along the gaps that paper names.

**Experiments**

| cell | what | gap it fills |
|---|---|---|
| **E0** | Fidelity calibration — `ControlMode` OFF vs STATIC against their reported baseline | their paper never states whether regulators act |
| **E1** | Reproduction of their Tables I & II, with per-bus / per-phase / magnitude metrics added | their metric is the feeder **mean** voltage |
| **E2** | **Multi-hub with realistic fleet constraints** | §III-A: *"assuming sufficient EV capacity at each hub"* — never tested |
| **E3** | **Degradation-aware objective** — weight sweep → violation/wear frontier | conclusion: *"battery-degradation-aware optimization"* |
| **E4** | **Multi-hub aggressive stress**, day-structured vs paper-style training | Table II: droop **2** viol-hours vs RL **15** |

**Design choices that make the numbers readable**

- *Common random numbers*: every controller in a comparison sees the identical availability
  realisation, initial SOC and load peak. Differences are paired, so scenario noise leaves
  the error bars instead of hiding the effect.
- *Integrated violation magnitude* (`IntViol`, p.u.·hours) alongside hour counts. Counts
  saturate — every controller can tie at "all 18 hours violated" — magnitudes do not.
- *Per-phase voltages* reported next to the feeder mean, since ANSI C84.1 is a per-phase
  limit and the feeder is unbalanced.

Run **QUICK = True** first (a few minutes) to check wiring, then set it False for the real run.


In [ ]:
# 0. dependencies (installs only what is missing)
# NOTE: the import name is `opendssdirect` but the PyPI distribution is `opendssdirect.py`.
import importlib, subprocess, sys

def ensure(mod, pkg):
    try:
        importlib.import_module(mod)
        return
    except ImportError:
        pass
    print(f"installing {pkg} ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:]);  print(r.stderr[-1500:])
        raise SystemExit(f"pip install {pkg} failed. Install it manually, then re-run this cell.")
    importlib.invalidate_caches()
    importlib.import_module(mod)

for mod, pkg in [("opendssdirect", "opendssdirect.py"),
                 ("gymnasium", "gymnasium"),
                 ("stable_baselines3", "stable-baselines3"),
                 ("matplotlib", "matplotlib")]:
    ensure(mod, pkg)

import numpy, torch
from importlib.metadata import version, PackageNotFoundError
try:
    dssver = version("opendssdirect.py")
except PackageNotFoundError:
    dssver = "?"
print(f"numpy {numpy.__version__} | torch {torch.__version__} | opendssdirect.py {dssver}")
print("deps OK")

### Materialize the feeder and modules (self-contained)

In [ ]:
%%writefile ieee34_master.dss
! Standard (Mod 1) model of IEEE 34 Bus Test Feeder

! Note: Mod 2 better accounts for distributed load.

Clear
Set DefaultBaseFrequency=60

New object=circuit.ieee34-1
~ basekv=69 pu=1.05 angle=30 mvasc3=200000  !stiffen up a bit over DSS default

! Substation Transformer  -- Modification: Make source very stiff by defining a tiny leakage Z
New Transformer.SubXF Phases=3 Windings=2 Xhl=0.01    ! normally 8
~ wdg=1 bus=sourcebus conn=Delta kv=69    kva=25000   %r=0.0005   !reduce %r, too
~ wdg=2 bus=800       conn=wye   kv=24.9  kva=25000   %r=0.0005

! import line codes with phase impedance matrices
Redirect IEEELineCodes.dss   ! revised according to Later test feeder doc

! Lines
New Line.L1     Phases=3 Bus1=800.1.2.3  Bus2=802.1.2.3  LineCode=300  Length=2.58   units=kft
New Line.L2     Phases=3 Bus1=802.1.2.3  Bus2=806.1.2.3  LineCode=300  Length=1.73   units=kft
New Line.L3     Phases=3 Bus1=806.1.2.3  Bus2=808.1.2.3  LineCode=300  Length=32.23   units=kft
New Line.L4     Phases=1 Bus1=808.2      Bus2=810.2      LineCode=303  Length=5.804   units=kft
New Line.L5     Phases=3 Bus1=808.1.2.3  Bus2=812.1.2.3  LineCode=300  Length=37.5   units=kft
New Line.L6     Phases=3 Bus1=812.1.2.3  Bus2=814.1.2.3  LineCode=300  Length=29.73   units=kft
New Line.L7     Phases=3 Bus1=814r.1.2.3 Bus2=850.1.2.3  LineCode=301  Length=0.01   units=kft
New Line.L8     Phases=1 Bus1=816.1      Bus2=818.1      LineCode=302  Length=1.71   units=kft
New Line.L9     Phases=3 Bus1=816.1.2.3  Bus2=824.1.2.3  LineCode=301  Length=10.21   units=kft
New Line.L10    Phases=1 Bus1=818.1      Bus2=820.1      LineCode=302  Length=48.15   units=kft
New Line.L11    Phases=1 Bus1=820.1      Bus2=822.1      LineCode=302  Length=13.74   units=kft
New Line.L12    Phases=1 Bus1=824.2      Bus2=826.2      LineCode=303  Length=3.03   units=kft
New Line.L13    Phases=3 Bus1=824.1.2.3  Bus2=828.1.2.3  LineCode=301  Length=0.84   units=kft
New Line.L14    Phases=3 Bus1=828.1.2.3  Bus2=830.1.2.3  LineCode=301  Length=20.44   units=kft
New Line.L15    Phases=3 Bus1=830.1.2.3  Bus2=854.1.2.3  LineCode=301  Length=0.52   units=kft
New Line.L16    Phases=3 Bus1=832.1.2.3  Bus2=858.1.2.3  LineCode=301  Length=4.9   units=kft
New Line.L17    Phases=3 Bus1=834.1.2.3  Bus2=860.1.2.3  LineCode=301  Length=2.02   units=kft
New Line.L18    Phases=3 Bus1=834.1.2.3  Bus2=842.1.2.3  LineCode=301  Length=0.28   units=kft
New Line.L19    Phases=3 Bus1=836.1.2.3  Bus2=840.1.2.3  LineCode=301  Length=0.86   units=kft
New Line.L20    Phases=3 Bus1=836.1.2.3  Bus2=862.1.2.3  LineCode=301  Length=0.28   units=kft
New Line.L21    Phases=3 Bus1=842.1.2.3  Bus2=844.1.2.3  LineCode=301  Length=1.35   units=kft
New Line.L22    Phases=3 Bus1=844.1.2.3  Bus2=846.1.2.3  LineCode=301  Length=3.64   units=kft
New Line.L23    Phases=3 Bus1=846.1.2.3  Bus2=848.1.2.3  LineCode=301  Length=0.53   units=kft
New Line.L24    Phases=3 Bus1=850.1.2.3  Bus2=816.1.2.3  LineCode=301  Length=0.31   units=kft
New Line.L25    Phases=3 Bus1=852r.1.2.3 Bus2=832.1.2.3  LineCode=301  Length=0.01   units=kft

! 24.9/4.16 kV  Transformer
New Transformer.XFM1  Phases=3 Windings=2 Xhl=4.08
~ wdg=1 bus=832       conn=wye   kv=24.9  kva=500    %r=0.95
~ wdg=2 bus=888       conn=Wye   kv=4.16  kva=500    %r=0.95

New Line.L26    Phases=1 Bus1=854.2      Bus2=856.2      LineCode=303  Length=23.33   units=kft
New Line.L27    Phases=3 Bus1=854.1.2.3  Bus2=852.1.2.3  LineCode=301  Length=36.83   units=kft
! 9-17-10 858-864 changed to phase A per error report
New Line.L28    Phases=1 Bus1=858.1      Bus2=864.1      LineCode=303  Length=1.62   units=kft
New Line.L29    Phases=3 Bus1=858.1.2.3  Bus2=834.1.2.3  LineCode=301  Length=5.83   units=kft
New Line.L30    Phases=3 Bus1=860.1.2.3  Bus2=836.1.2.3  LineCode=301  Length=2.68   units=kft
New Line.L31    Phases=1 Bus1=862.2      Bus2=838.2      LineCode=304  Length=4.86   units=kft
New Line.L32    Phases=3 Bus1=888.1.2.3  Bus2=890.1.2.3  LineCode=300  Length=10.56   units=kft

! Capacitors
New Capacitor.C844      Bus1=844        Phases=3        kVAR=300        kV=24.9
New Capacitor.C848      Bus1=848        Phases=3        kVAR=450        kV=24.9

! Regulators - three independent phases
! Regulator 1
new transformer.reg1a phases=1 windings=2 bank=reg1 buses=(814.1 814r.1) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg1a transformer=reg1a winding=2 vreg=122 band=2 ptratio=120 ctprim=100 R=2.7 X=1.6
new transformer.reg1b phases=1 windings=2 bank=reg1 buses=(814.2 814r.2) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg1b transformer=reg1b winding=2 vreg=122 band=2 ptratio=120 ctprim=100 R=2.7 X=1.6
new transformer.reg1c phases=1 windings=2 bank=reg1 buses=(814.3 814r.3) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg1c transformer=reg1c winding=2 vreg=122 band=2 ptratio=120 ctprim=100 R=2.7 X=1.6

! Regulator 2
new transformer.reg2a phases=1 windings=2 bank=reg2 buses=(852.1 852r.1) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg2a transformer=reg2a winding=2 vreg=124 band=2 ptratio=120 ctprim=100 R=2.5 X=1.5
new transformer.reg2b phases=1 windings=2 bank=reg2 buses=(852.2 852r.2) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg2b transformer=reg2b winding=2 vreg=124 band=2 ptratio=120 ctprim=100 R=2.5 X=1.5
new transformer.reg2c phases=1 windings=2 bank=reg2 buses=(852.3 852r.3) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg2c transformer=reg2c winding=2 vreg=124 band=2 ptratio=120 ctprim=100 R=2.5 X=1.5

! spot loads
New Load.S860       Bus1=860   Phases=3 Conn=Wye   Model=1 kV= 24.900 kW=  60.0 kVAR=  48.0
New Load.S840       Bus1=840   Phases=3 Conn=Wye   Model=5 kV= 24.900 kW=  27.0 kVAR=  21.0
New Load.S844       Bus1=844   Phases=3 Conn=Wye   Model=2 kV= 24.900 kW= 405.0 kVAR= 315.0

New Load.S848       Bus1=848   Phases=3 Conn=Delta Model=1 kV= 24.900 kW=  60.0 kVAR=  48.0
New Load.S830a      Bus1=830.1.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  10.0 kVAR=   5.0
New Load.S830b      Bus1=830.2.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  10.0 kVAR=   5.0
New Load.S830c      Bus1=830.3.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  25.0 kVAR=  10.0
New Load.S890       Bus1=890   Phases=3 Conn=Delta Model=5 kV=  4.160 kW= 450.0 kVAR= 225.0

! distributed loads
New Load.D802_806sb Bus1=802.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  15.0 kVAR=   7.5
New Load.D802_806rb Bus1=806.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  15.0 kVAR=   7.5
New Load.D802_806sc Bus1=802.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  12.5 kVAR=   7.0
New Load.D802_806rc Bus1=806.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  12.5 kVAR=   7.0

New Load.D808_810sb Bus1=808.2 Phases=1 Conn=Wye   Model=4 kV= 14.376 kW=   8.0 kVAR=   4.0
New Load.D808_810rb Bus1=810.2 Phases=1 Conn=Wye   Model=4 kV= 14.376 kW=   8.0 kVAR=   4.0

New Load.D818_820sa Bus1=818.1 Phases=1 Conn=Wye   Model=2 kV= 14.376 kW=  17.0 kVAR=   8.5
New Load.D818_820ra Bus1=820.1 Phases=1 Conn=Wye   Model=2 kV= 14.376 kW=  17.0 kVAR=   8.5

New Load.D820_822sa Bus1=820.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  67.5 kVAR=  35.0
New Load.D820_822ra Bus1=822.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  67.5 kVAR=  35.0

New Load.D816_824sb Bus1=816.2.3 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=   2.5 kVAR=   1.0
New Load.D816_824rb Bus1=824.2.3 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=   2.5 kVAR=   1.0

New Load.D824_826sb Bus1=824.2 Phases=1 Conn=Wye   Model=5 kV= 14.376 kW=  20.0 kVAR=  10.0
New Load.D824_826rb Bus1=826.2 Phases=1 Conn=Wye   Model=5 kV= 14.376 kW=  20.0 kVAR=  10.0
New Load.D824_828sc Bus1=824.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   2.0 kVAR=   1.0
New Load.D824_828rc Bus1=828.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   2.0 kVAR=   1.0

New Load.D828_830sa Bus1=828.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   3.5 kVAR=   1.5
New Load.D828_830ra Bus1=830.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   3.5 kVAR=   1.5

New Load.D854_856sb Bus1=854.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   2.0 kVAR=   1.0
New Load.D854_856rb Bus1=856.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   2.0 kVAR=   1.0

New Load.D832_858sa Bus1=832.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   3.5 kVAR=   1.5
New Load.D832_858ra Bus1=858.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   3.5 kVAR=   1.5
New Load.D832_858sb Bus1=832.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   1.0 kVAR=   0.5
New Load.D832_858rb Bus1=858.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   1.0 kVAR=   0.5
New Load.D832_858sc Bus1=832.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   3.0 kVAR=   1.5
New Load.D832_858rc Bus1=858.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   3.0 kVAR=   1.5

! 9-17-10 858-864 changed to phase A per error report
New Load.D858_864sb Bus1=858.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   1.0 kVAR=   0.5
New Load.D858_864rb Bus1=864.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   1.0 kVAR=   0.5

New Load.D858_834sa Bus1=858.1.2 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   2.0 kVAR=   1.0
New Load.D858_834ra Bus1=834.1.2 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   2.0 kVAR=   1.0
New Load.D858_834sb Bus1=858.2.3 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   7.5 kVAR=   4.0
New Load.D858_834rb Bus1=834.2.3 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   7.5 kVAR=   4.0
New Load.D858_834sc Bus1=858.3.1 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   6.5 kVAR=   3.5
New Load.D858_834rc Bus1=834.3.1 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   6.5 kVAR=   3.5

New Load.D834_860sa Bus1=834.1.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   8.0 kVAR=   4.0
New Load.D834_860ra Bus1=860.1.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   8.0 kVAR=   4.0
New Load.D834_860sb Bus1=834.2.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  10.0 kVAR=   5.0
New Load.D834_860rb Bus1=860.2.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  10.0 kVAR=   5.0
New Load.D834_860sc Bus1=834.3.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  55.0 kVAR=  27.5
New Load.D834_860rc Bus1=860.3.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  55.0 kVAR=  27.5

New Load.D860_836sa Bus1=860.1.2 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=  15.0 kVAR=   7.5
New Load.D860_836ra Bus1=836.1.2 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=  15.0 kVAR=   7.5
New Load.D860_836sb Bus1=860.2.3 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   5.0 kVAR=   3.0
New Load.D860_836rb Bus1=836.2.3 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   5.0 kVAR=   3.0
New Load.D860_836sc Bus1=860.3.1 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=  21.0 kVAR=  11.0
New Load.D860_836rc Bus1=836.3.1 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=  21.0 kVAR=  11.0

New Load.D836_840sa Bus1=836.1.2 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=   9.0 kVAR=   4.5
New Load.D836_840ra Bus1=840.1.2 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=   9.0 kVAR=   4.5
New Load.D836_840sb Bus1=836.2.3 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=  11.0 kVAR=   5.5
New Load.D836_840rb Bus1=840.2.3 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=  11.0 kVAR=   5.5

New Load.D862_838sb Bus1=862.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  14.0 kVAR=   7.0
New Load.D862_838rb Bus1=838.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  14.0 kVAR=   7.0

New Load.D842_844sa Bus1=842.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   4.5 kVAR=   2.5
New Load.D842_844ra Bus1=844.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   4.5 kVAR=   2.5

New Load.D844_846sb Bus1=844.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  12.5 kVAR=   6.0
New Load.D844_846rb Bus1=846.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  12.5 kVAR=   6.0
New Load.D844_846sc Bus1=844.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  10.0 kVAR=   5.5
New Load.D844_846rc Bus1=846.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  10.0 kVAR=   5.5

New Load.D846_848sb Bus1=846.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  11.5 kVAR=   5.5
New Load.D846_848rb Bus1=848.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  11.5 kVAR=   5.5

! Script to revise Vminpu property on all loads to allow voltage to sag to 85% without switching
! to constant Z model
Load.s860.vminpu=.85
Load.s840.vminpu=.85
Load.s844.vminpu=.85
Load.s848.vminpu=.85
Load.s830a.vminpu=.85
Load.s830b.vminpu=.85
Load.s830c.vminpu=.85
Load.s890.vminpu=.85
Load.d802_806sb.vminpu=.85
Load.d802_806rb.vminpu=.85
Load.d802_806sc.vminpu=.85
Load.d802_806rc.vminpu=.85
Load.d808_810sb.vminpu=.85
Load.d808_810rb.vminpu=.85
Load.d818_820sa.vminpu=.85
Load.d818_820ra.vminpu=.85
Load.d820_822sa.vminpu=.85
Load.d820_822ra.vminpu=.85
Load.d816_824sb.vminpu=.85
Load.d816_824rb.vminpu=.85
Load.d824_826sb.vminpu=.85
Load.d824_826rb.vminpu=.85
Load.d824_828sc.vminpu=.85
Load.d824_828rc.vminpu=.85
Load.d828_830sa.vminpu=.85
Load.d828_830ra.vminpu=.85
Load.d854_856sb.vminpu=.85
Load.d854_856rb.vminpu=.85
Load.d832_858sa.vminpu=.85
Load.d832_858ra.vminpu=.85
Load.d832_858sb.vminpu=.85
Load.d832_858rb.vminpu=.85
Load.d832_858sc.vminpu=.85
Load.d832_858rc.vminpu=.85
Load.d858_864sb.vminpu=.85
Load.d858_864rb.vminpu=.85
Load.d858_834sa.vminpu=.85
Load.d858_834ra.vminpu=.85
Load.d858_834sb.vminpu=.85
Load.d858_834rb.vminpu=.85
Load.d858_834sc.vminpu=.85
Load.d858_834rc.vminpu=.85
Load.d834_860sa.vminpu=.85
Load.d834_860ra.vminpu=.85
Load.d834_860sb.vminpu=.85
Load.d834_860rb.vminpu=.85
Load.d834_860sc.vminpu=.85
Load.d834_860rc.vminpu=.85
Load.d860_836sa.vminpu=.85
Load.d860_836ra.vminpu=.85
Load.d860_836sb.vminpu=.85
Load.d860_836rb.vminpu=.85
Load.d860_836sc.vminpu=.85
Load.d860_836rc.vminpu=.85
Load.d836_840sa.vminpu=.85
Load.d836_840ra.vminpu=.85
Load.d836_840sb.vminpu=.85
Load.d836_840rb.vminpu=.85
Load.d862_838sb.vminpu=.85
Load.d862_838rb.vminpu=.85
Load.d842_844sa.vminpu=.85
Load.d842_844ra.vminpu=.85
Load.d844_846sb.vminpu=.85
Load.d844_846rb.vminpu=.85
Load.d844_846sc.vminpu=.85
Load.d844_846rc.vminpu=.85
Load.d846_848sb.vminpu=.85
Load.d846_848rb.vminpu=.85


! let the DSS estimate voltage bases automatically
Set VoltageBases = "69,24.9,4.16, .48"
CalcVoltageBases


In [ ]:
%%writefile IEEELineCodes.dss
! this file was corrected 9/16/2010 to match the values in Kersting's files



! These line codes are used in the 123-bus circuit

New linecode.1 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0312137 0.0901946 | 0.0306264 0.0316143 0.0889665 )
!!!~ xmatrix = (0.20744 | 0.0935314 0.200783 | 0.0760312 0.0855879 0.204877 )
!!!~ cmatrix = (2.90301 | -0.679335 3.15896 | -0.22313 -0.481416 2.8965 )
~ rmatrix = [0.086666667 | 0.029545455 0.088371212 | 0.02907197 0.029924242 0.087405303]
~ xmatrix = [0.204166667 | 0.095018939 0.198522727 | 0.072897727 0.080227273 0.201723485]
~ cmatrix = [2.851710072 | -0.920293787  3.004631862 | -0.350755566  -0.585011253 2.71134756]

New linecode.2 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0901946 | 0.0316143 0.0889665 | 0.0312137 0.0306264 0.088205 )
!!!~ xmatrix = (0.200783 | 0.0855879 0.204877 | 0.0935314 0.0760312 0.20744 )
!!!~ cmatrix = (3.15896 | -0.481416 2.8965 | -0.679335 -0.22313 2.90301 )
~ rmatrix = [0.088371212 | 0.02992424  0.087405303 | 0.029545455 0.02907197 0.086666667]
~ xmatrix = [0.198522727 | 0.080227273  0.201723485 | 0.095018939 0.072897727 0.204166667]
~ cmatrix = [3.004631862 | -0.585011253 2.71134756 | -0.920293787  -0.350755566  2.851710072]

New linecode.3 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0889665 | 0.0306264 0.088205 | 0.0316143 0.0312137 0.0901946 )
!!!~ xmatrix = (0.204877 | 0.0760312 0.20744 | 0.0855879 0.0935314 0.200783 )
!!!~ cmatrix = (2.8965 | -0.22313 2.90301 | -0.481416 -0.679335 3.15896 )

~ rmatrix = [0.087405303 | 0.02907197 0.086666667  | 0.029924242 0.029545455 0.088371212]
~ xmatrix = [0.201723485 | 0.072897727 0.204166667 | 0.080227273 0.095018939 0.198522727]
~ cmatrix = [2.71134756  | -0.350755566 2.851710072 | -0.585011253 -0.920293787 3.004631862]

New linecode.4 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0889665 | 0.0316143 0.0901946 | 0.0306264 0.0312137 0.088205 )
!!!~ xmatrix = (0.204877 | 0.0855879 0.200783 | 0.0760312 0.0935314 0.20744 )
!!!~ cmatrix = (2.8965 | -0.481416 3.15896 | -0.22313 -0.679335 2.90301 )
~ rmatrix = [0.087405303 | 0.029924242 0.088371212 | 0.02907197   0.029545455 0.086666667]
~ xmatrix = [0.201723485 | 0.080227273 0.198522727 | 0.072897727 0.095018939 0.204166667]
~ cmatrix = [2.71134756  | -0.585011253 3.004631862 | -0.350755566 -0.920293787 2.851710072]

New linecode.5 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0901946 | 0.0312137 0.088205 | 0.0316143 0.0306264 0.0889665 )
!!!~ xmatrix = (0.200783 | 0.0935314 0.20744 | 0.0855879 0.0760312 0.204877 )
!!!~ cmatrix = (3.15896 | -0.679335 2.90301 | -0.481416 -0.22313 2.8965 )

~ rmatrix = [0.088371212  |  0.029545455  0.086666667  |  0.029924242  0.02907197  0.087405303]
~ xmatrix = [0.198522727  |  0.095018939  0.204166667  |  0.080227273  0.072897727  0.201723485]
~ cmatrix = [3.004631862  | -0.920293787  2.851710072  |  -0.585011253  -0.350755566  2.71134756]

New linecode.6 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0306264 0.0889665 | 0.0312137 0.0316143 0.0901946 )
!!!~ xmatrix = (0.20744 | 0.0760312 0.204877 | 0.0935314 0.0855879 0.200783 )
!!!~ cmatrix = (2.90301 | -0.22313 2.8965 | -0.679335 -0.481416 3.15896 )
~ rmatrix = [0.086666667 | 0.02907197  0.087405303 | 0.029545455  0.029924242  0.088371212]
~ xmatrix = [0.204166667 | 0.072897727  0.201723485 | 0.095018939  0.080227273  0.198522727]
~ cmatrix = [2.851710072 | -0.350755566  2.71134756 | -0.920293787  -0.585011253  3.004631862]
New linecode.7 nphases=2 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0306264 0.0889665 )
!!!~ xmatrix = (0.20744 | 0.0760312 0.204877 )
!!!~ cmatrix = (2.75692 | -0.326659 2.82313 )
~ rmatrix = [0.086666667 | 0.02907197  0.087405303]
~ xmatrix = [0.204166667 | 0.072897727  0.201723485]
~ cmatrix = [2.569829596 | -0.52995137  2.597460011]
New linecode.8 nphases=2 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0306264 0.0889665 )
!!!~ xmatrix = (0.20744 | 0.0760312 0.204877 )
!!!~ cmatrix = (2.75692 | -0.326659 2.82313 )
~ rmatrix = [0.086666667 | 0.02907197  0.087405303]
~ xmatrix = [0.204166667 | 0.072897727  0.201723485]
~ cmatrix = [2.569829596 | -0.52995137  2.597460011]
New linecode.9 nphases=1 BaseFreq=60 units=kft
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.10 nphases=1 BaseFreq=60 units=kft
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.11 nphases=1 BaseFreq=60 units=kft
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.12 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.291814 | 0.101656 0.294012 | 0.096494 0.101656 0.291814 )
!!!~ xmatrix = (0.141848 | 0.0517936 0.13483 | 0.0401881 0.0517936 0.141848 )
!!!~ cmatrix = (53.4924 | 0 53.4924 | 0 0 53.4924 )
~ rmatrix = [0.288049242 | 0.09844697  0.29032197 | 0.093257576  0.09844697  0.288049242]
~ xmatrix = [0.142443182 | 0.052556818  0.135643939 | 0.040852273  0.052556818  0.142443182]
~ cmatrix = [33.77150149 | 0  33.77150149 | 0  0  33.77150149]

! These line codes are used in the 34-node test feeder

New linecode.300 nphases=3 basefreq=60   units=kft   ! ohms per 1000ft  Corrected 11/30/05
~ rmatrix = [0.253181818   |  0.039791667     0.250719697  |   0.040340909      0.039128788     0.251780303]  !ABC ORDER
~ xmatrix = [0.252708333   |  0.109450758     0.256988636  |   0.094981061      0.086950758     0.255132576]
~ CMATRIX = [2.680150309   | -0.769281006     2.5610381    |  -0.499507676     -0.312072984     2.455590387]
New linecode.301 nphases=3 basefreq=60   units=kft
~ rmatrix = [0.365530303   |   0.04407197      0.36282197   |   0.04467803       0.043333333     0.363996212]
~ xmatrix = [0.267329545   |   0.122007576     0.270473485  |   0.107784091      0.099204545     0.269109848] 
~ cmatrix = [2.572492163   |  -0.72160598      2.464381882  |  -0.472329395     -0.298961096     2.368881119]
New linecode.302 nphases=1 basefreq=60   units=kft
~ rmatrix = (0.530208 )
~ xmatrix = (0.281345 )
~ cmatrix = (2.12257 )
New linecode.303 nphases=1 basefreq=60   units=kft
~ rmatrix = (0.530208 )
~ xmatrix = (0.281345 )
~ cmatrix = (2.12257 )
New linecode.304 nphases=1 basefreq=60   units=kft
~ rmatrix = (0.363958 )
~ xmatrix = (0.269167 )
~ cmatrix = (2.1922 )


! This may be for the 4-node test feeder, but is not actually referenced.
!  instead, the 4Bus*.dss files all use the wiredata and linegeometry inputs
!  to calculate these matrices from physical data.

New linecode.400 nphases=3 BaseFreq=60
~ rmatrix = (0.088205 | 0.0312137 0.0901946 | 0.0306264 0.0316143 0.0889665 )
~ xmatrix = (0.20744 | 0.0935314 0.200783 | 0.0760312 0.0855879 0.204877 )
~ cmatrix = (2.90301 | -0.679335 3.15896 | -0.22313 -0.481416 2.8965 )

! These are for the 13-node test feeder

New linecode.601 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.0674673 | 0.0312137 0.0654777 | 0.0316143 0.0306264 0.0662392 )
!!!~ xmatrix = (0.195204  | 0.0935314 0.201861 | 0.0855879 0.0760312 0.199298 )
!!!~ cmatrix = (3.32591   | -0.743055 3.04217 | -0.525237 -0.238111 3.03116 )
~ rmatrix = [0.065625    | 0.029545455  0.063920455  | 0.029924242  0.02907197  0.064659091]
~ xmatrix = [0.192784091 | 0.095018939  0.19844697   | 0.080227273  0.072897727  0.195984848]
~ cmatrix = [3.164838036 | -1.002632425  2.993981593 | -0.632736516  -0.372608713  2.832670203]
New linecode.602 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.144361 | 0.0316143 0.143133 | 0.0312137 0.0306264 0.142372 )
!!!~ xmatrix = (0.226028 | 0.0855879 0.230122 | 0.0935314 0.0760312 0.232686 )
!!!~ cmatrix = (3.01091  | -0.443561 2.77543  | -0.624494 -0.209615 2.77847 )
~ rmatrix = [0.142537879 | 0.029924242  0.14157197   | 0.029545455  0.02907197  0.140833333]
~ xmatrix = [0.22375     | 0.080227273  0.226950758  | 0.095018939  0.072897727  0.229393939]
~ cmatrix = [2.863013423 | -0.543414918  2.602031589 | -0.8492585  -0.330962141  2.725162768]
New linecode.603 nphases=2 BaseFreq=60
!!!~ rmatrix = (0.254472 | 0.0417943 0.253371 )
!!!~ xmatrix = (0.259467 | 0.0912376 0.261431 )
!!!~ cmatrix = (2.54676  | -0.28882 2.49502 )
~ rmatrix = [0.251780303 | 0.039128788  0.250719697]
~ xmatrix = [0.255132576 | 0.086950758  0.256988636]
~ cmatrix = [2.366017603 | -0.452083836  2.343963508]
New linecode.604 nphases=2 BaseFreq=60
!!!~ rmatrix = (0.253371 | 0.0417943 0.254472 )
!!!~ xmatrix = (0.261431 | 0.0912376 0.259467 )
!!!~ cmatrix = (2.49502 | -0.28882 2.54676 )
~ rmatrix = [0.250719697 | 0.039128788   0.251780303]
~ xmatrix = [0.256988636  | 0.086950758  0.255132576]
~ cmatrix = [2.343963508 | -0.452083836 2.366017603]
New linecode.605 nphases=1 BaseFreq=60
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.606 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.152193 | 0.0611362 0.15035 | 0.0546992 0.0611362 0.152193 )
!!!~ xmatrix = (0.0825685 | 0.00548281 0.0745027 | -0.00339824 0.00548281 0.0825685 )
!!!~ cmatrix = (72.7203 | 0 72.7203 | 0 0 72.7203 )
~ rmatrix = [0.151174242 | 0.060454545  0.149450758 | 0.053958333  0.060454545  0.151174242]
~ xmatrix = [0.084526515 | 0.006212121  0.076534091 | -0.002708333  0.006212121  0.084526515]
~ cmatrix = [48.67459408 | 0  48.67459408 | 0  0  48.67459408]
New linecode.607 nphases=1 BaseFreq=60
!!!~ rmatrix = (0.255799 )
!!!~ xmatrix = (0.092284 )
!!!~ cmatrix = (50.7067 )
~ rmatrix = [0.254261364]
~ xmatrix = [0.097045455]
~ cmatrix = [44.70661522]

! These are for the 37-node test feeder, all underground

New linecode.721 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.0554906 | 0.0127467 0.0501597 | 0.00640446 0.0127467 0.0554906 )
!!!~ xmatrix = (0.0372331 | -0.00704588 0.0358645 | -0.00796424 -0.00704588 0.0372331 )
!!!~ cmatrix = (124.851 | 0 124.851 | 0 0 124.851 )
~ rmatrix = [0.055416667 | 0.012746212  0.050113636  | 0.006382576  0.012746212  0.055416667]
~ xmatrix = [0.037367424 | -0.006969697  0.035984848 | -0.007897727  -0.006969697  0.037367424]
~ cmatrix = [80.27484728 | 0  80.27484728            | 0  0  80.27484728]
New linecode.722 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.0902251 | 0.0309584 0.0851482 | 0.0234946 0.0309584 0.0902251 )
!!!~ xmatrix = (0.055991 | -0.00646552 0.0504025 | -0.0117669 -0.00646552 0.055991 )
!!!~ cmatrix = (93.4896 | 0 93.4896 | 0 0 93.4896 )
~ rmatrix = [0.089981061 | 0.030852273  0.085        | 0.023371212  0.030852273  0.089981061]
~ xmatrix = [0.056306818 | -0.006174242  0.050719697 | -0.011496212  -0.006174242  0.056306818]
~ cmatrix = [64.2184109 | 0  64.2184109              | 0  0  64.2184109]
New linecode.723 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.247572 | 0.0947678 0.249104 | 0.0893782 0.0947678 0.247572 )
!!!~ xmatrix = (0.126339 | 0.0390337 0.118816 | 0.0279344 0.0390337 0.126339 )
!!!~ cmatrix = (58.108 | 0 58.108 | 0 0 58.108 )
~ rmatrix = [0.245 | 0.092253788  0.246628788 | 0.086837121  0.092253788  0.245]
~ xmatrix = [0.127140152 | 0.039981061  0.119810606 | 0.028806818  0.039981061  0.127140152]
~ cmatrix = [37.5977112 | 0  37.5977112 | 0  0  37.5977112]
New linecode.724 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.399883 | 0.101765 0.402011 | 0.0965199 0.101765 0.399883 )
!!!~ xmatrix = (0.146325 | 0.0510963 0.139305 | 0.0395402 0.0510963 0.146325 )
!!!~ cmatrix = (46.9685 | 0 46.9685 | 0 0 46.9685 )
~ rmatrix = [0.396818182 | 0.098560606  0.399015152 | 0.093295455  0.098560606  0.396818182]
~ xmatrix = [0.146931818 | 0.051856061  0.140113636 | 0.040208333  0.051856061  0.146931818]
~ cmatrix = [30.26701029 | 0  30.26701029 | 0  0  30.26701029]


In [ ]:
%%writefile v2g_sys.py
"""
System layer for the V2G gap study — paper-faithful corrections over v2g_core.py.

Differences from the original reproduction, all deliberate:
  * Feeder(control_mode=...)   : regulators can be STATIC (OpenDSS default, they act)
                                 or OFF (frozen). The paper never states which; E0 decides
                                 it empirically from their reported baseline fingerprint.
  * droop_pq(sat=0.90/1.10)    : matches the paper's stated saturation. The earlier code
                                 used 0.94/1.06, which made droop noticeably stronger.
  * per-PHASE voltages          : ANSI C84.1 limits are per-phase. We keep every energized
                                 node voltage instead of averaging phases per bus.
  * stochastic availability     : n_avail is a binomial draw around the mean profile, so
                                 scenario variance is explicit (and can be paired away).
  * throughput accounting       : cumulative battery kWh, for the degradation objective.
"""
import os
import numpy as np
import opendssdirect as dss

MASTER = os.path.abspath("ieee34_master.dss")

CFG = dict(
    v_min=0.95, v_max=1.05,
    hub_buses_multi=["890", "844", "832", "830", "860"],
    hub_bus_single=["890"],
    P_rated=500.0, Q_rated=400.0,               # kW / kVAr per hub
    ev_capacity=75.0, soc_init=0.7, soc_min=0.2, soc_max=0.9,
    soh=0.95, eta_inv=0.96, c_rate=0.5, n_ev=15,
    active_hours=list(range(6, 24)),            # 06:00 .. 23:00
    peak_mild=1.5, peak_aggr=3.0,
)

# Normalised daily load shape (peak 1.0 near 18:00).
LOAD_SHAPE = np.array([0.24, 0.22, 0.20, 0.20, 0.22, 0.26, 0.30, 0.42, 0.55, 0.63,
                       0.70, 0.74, 0.77, 0.79, 0.82, 0.86, 0.92, 0.97, 1.00, 0.96,
                       0.86, 0.66, 0.44, 0.30])

# Mean EV participation. Paper states 45-85% for the single-hub fleet.
AVAIL_MEAN = np.array([0.85, 0.85, 0.85, 0.85, 0.80, 0.75, 0.65, 0.55, 0.50, 0.47,
                       0.45, 0.45, 0.48, 0.50, 0.55, 0.60, 0.65, 0.72, 0.78, 0.82,
                       0.85, 0.85, 0.85, 0.85])

BASE_LL_KV = {"890": 4.16, "888": 4.16}   # 4.16 kV transformer secondary


def lam_profile(peak):
    return LOAD_SHAPE * peak


# --------------------------------------------------------------------------- #
# Feeder
# --------------------------------------------------------------------------- #
class Feeder:
    """IEEE-34 wrapper exposing per-bus and per-phase voltages."""

    def __init__(self, hub_buses, control_mode="OFF"):
        self.hub_buses = list(hub_buses)
        self.control_mode = control_mode
        dss.Command("Clear")
        dss.Command(f'Compile "{MASTER}"')
        dss.Command(f"Set ControlMode={control_mode}")
        dss.Command("Set MaxIterations=100")
        dss.Command("CalcVoltageBases")
        dss.Command("Solve")
        self.buses = [b for b in dss.Circuit.AllBusNames() if b.lower() != "sourcebus"]
        self.hub_kv = {}
        for b in self.hub_buses:
            dss.Circuit.SetActiveBus(b)
            kvb = dss.Bus.kVBase()
            kv = round(kvb * np.sqrt(3), 3) if kvb > 0.1 else BASE_LL_KV.get(b.lower(), 24.9)
            self.hub_kv[b] = kv
            dss.Command(f"New Generator.hub{b} bus1={b}.1.2.3 phases=3 kv={kv} "
                        f"kw=0 kvar=0 model=1 Vminpu=0.5 Vmaxpu=1.5 status=fixed")
        dss.Command("Solve")

    # ---- actuation ----
    def set_load(self, lam):
        dss.Command(f"Set LoadMult={lam}")

    def set_hub(self, bus, p_kw, q_kvar):
        dss.Command(f"Generator.hub{bus}.kW={p_kw}")
        dss.Command(f"Generator.hub{bus}.kvar={q_kvar}")

    def zero_hubs(self):
        for b in self.hub_buses:
            self.set_hub(b, 0.0, 0.0)

    def solve(self):
        dss.Command("Solve")
        return dss.Solution.Converged()

    # ---- measurement ----
    def phase_vpu(self):
        """Every energized node (per-phase) voltage in p.u. — the ANSI-relevant set."""
        out = []
        for b in self.buses:
            dss.Circuit.SetActiveBus(b)
            out.extend(v for v in dss.Bus.puVmagAngle()[0::2] if v > 0.01)
        return np.asarray(out)

    def bus_vpu(self):
        """Per-bus voltage, averaged over that bus's energized phases."""
        out = np.empty(len(self.buses))
        for i, b in enumerate(self.buses):
            dss.Circuit.SetActiveBus(b)
            vs = [v for v in dss.Bus.puVmagAngle()[0::2] if v > 0.01]
            out[i] = np.mean(vs) if vs else 1.0
        return out

    def hub_vpu(self, bus):
        dss.Circuit.SetActiveBus(bus)
        vs = [v for v in dss.Bus.puVmagAngle()[0::2] if v > 0.01]
        return float(np.mean(vs)) if vs else 1.0

    def feeder_mean(self):
        return float(np.mean(self.bus_vpu()))

    def tap_positions(self):
        """Regulator tap positions — non-trivial only when control_mode=STATIC."""
        taps = []
        names = dss.RegControls.AllNames()
        if not names or names == ['NONE']:
            return taps
        for n in names:
            dss.RegControls.Name(n)
            taps.append(dss.RegControls.TapNumber())
        return taps


# --------------------------------------------------------------------------- #
# Droop baseline — paper-stated deadband and saturation
# --------------------------------------------------------------------------- #
def droop_pq(v, P_rated, Q_rated, db=0.02, sat_lo=0.90, sat_hi=1.10):
    """Piecewise-linear Volt-Watt / Volt-Var. Positive P = discharge (support)."""
    if v < 1 - db:
        f = min(1.0, (1 - db - v) / (1 - db - sat_lo))
    elif v > 1 + db:
        f = -min(1.0, (v - (1 + db)) / (sat_hi - (1 + db)))
    else:
        f = 0.0
    return f * P_rated, f * Q_rated


# --------------------------------------------------------------------------- #
# EV fleet
# --------------------------------------------------------------------------- #
class EVFleet:
    """Aggregate hub fleet: availability-limited power, SOC state, throughput log."""

    def __init__(self, cfg=CFG, avail_scale=1.0, soc_on="S"):
        """soc_on: "S" drains the battery on APPARENT power, per the paper's Eq. (4)
        (P_fleet = S_req / eta_inv). "P" drains on real power only -- which leaves
        reactive support free and unlimited, and lets a learned policy hold voltage
        with Q at zero SOC cost. Kept as a switch so the sensitivity can be reported."""
        self.c = cfg
        self.avail_scale = avail_scale
        self.soc_on = soc_on
        self.reset()

    def reset(self, n_avail_day=None, soc_init=None):
        """n_avail_day: length-24 int array of available EVs (paired scenario draw)."""
        self.soc = self.c["soc_init"] if soc_init is None else float(soc_init)
        self.soh = self.c["soh"]
        self.throughput = 0.0          # cumulative battery kWh moved
        self.soc_series = [self.soc]
        self._n_day = n_avail_day

    def n_avail(self, hour):
        if self._n_day is not None:
            return int(self._n_day[hour])
        frac = np.clip(AVAIL_MEAN[hour] * self.avail_scale, 0.05, 1.0)
        return max(1, int(round(self.c["n_ev"] * frac)))

    def avail_power(self, hour):
        """Max discharge power this 1-h step: min(C-rate limit, usable-energy limit)."""
        n = self.n_avail(hour)
        cap, soh = self.c["ev_capacity"], self.soh
        p_crate = n * self.c["c_rate"] * cap
        e_usable = n * max(0.0, self.soc - self.c["soc_min"]) * cap * soh
        return min(p_crate, e_usable)

    def apply(self, p_grid, q_grid, hour, commit=True):
        """Scale requested (p, q) to fleet capability. Returns (p_sup, q_sup, rho, n)."""
        s_req = float(np.hypot(p_grid, q_grid))
        p_fleet = s_req / self.c["eta_inv"]
        p_avail = self.avail_power(hour)
        rho = min(1.0, p_avail / p_fleet) if p_fleet > 1e-6 else 1.0
        p_sup, q_sup = rho * p_grid, rho * q_grid
        if not commit:
            return p_sup, q_sup, rho, self.n_avail(hour)

        n = self.n_avail(hour)
        cap_tot = n * self.c["ev_capacity"] * self.soh
        eta = self.c["eta_inv"]
        s_sup = float(np.hypot(p_sup, q_sup))
        if p_sup >= 0:                       # net real-power export -> discharging
            e_out = (s_sup if self.soc_on == "S" else abs(p_sup)) / eta
            e_in = 0.0
        else:                                # charging; reactive support still costs
            e_in = abs(p_sup) * eta
            e_out = (abs(q_sup) / eta) if self.soc_on == "S" else 0.0
        dsoc = (e_in - e_out) / cap_tot
        self.throughput += (e_in + e_out) * 1.0            # battery kWh moved this hour
        self.soc = float(np.clip(self.soc + dsoc, self.c["soc_min"], self.c["soc_max"]))
        self.soc_series.append(self.soc)
        return p_sup, q_sup, rho, n


def draw_availability(rng, n_ev, avail_scale=1.0):
    """Binomial availability realisation for one day — the paired scenario primitive."""
    p = np.clip(AVAIL_MEAN * avail_scale, 0.02, 1.0)
    return np.maximum(1, rng.binomial(n_ev, p))


# --------------------------------------------------------------------------- #
# Voltage reward (paper Eqs. 8-10) — unchanged, so the baseline is comparable
# --------------------------------------------------------------------------- #
def reward_from_v(vpu, vmin=0.95, vmax=1.05):
    inb = bool(np.all((vpu >= vmin) & (vpu <= vmax)))
    Rvb = 10.0 if inb else 0.0
    pen = float(np.where(vpu < vmin, (vmin - vpu) * 100,
                np.where(vpu > vmax, (vpu - vmax) * 100, 0.0)).sum())
    return Rvb - pen


In [ ]:
%%writefile v2g_metrics.py
"""
Metric set for the V2G gap study.

Reports the paper's metric *and* the standard-compliant ones side by side, so the
two can be compared directly rather than argued about:

  ViolMean  hours where the FEEDER-MEAN voltage is out of band   <- the paper's metric
  ViolBus   hours where ANY bus (phase-averaged) is out of band
  ViolPh    hours where ANY energized phase is out of band       <- ANSI C84.1
  ViolHi    hours with an OVERvoltage specifically
  IntViol   integrated violation magnitude, p.u.-hours (= IntLo + IntHi)
  IntLo/IntHi   the under- and over-voltage halves of IntViol
  VMean/VMin/VMax   feeder-mean voltage stats, matching the paper's table columns
  VphMin/VphMax     worst single-phase voltage extremes over the day

Two properties this set is built for:

1. ALL VIOLATION METRICS ARE TWO-SIDED. A one-sided (undervoltage-only) metric scores an
   agent that shoves the feeder above 1.05 as violation-free -- which is exactly what an
   unconstrained learned policy will do when reactive power is cheap.
2. Counts saturate (every controller can tie at "all 18 hours violated"); IntViol does
   not, so it still separates controllers when the counts agree.
"""
import numpy as np

V_MIN, V_MAX = 0.95, 1.05
# Buses can sit exactly on a limit (e.g. a regulated bus pinned at 1.05 p.u.). Without a
# tolerance, floating-point noise flags every such hour as a violation while the integrated
# magnitude stays 0.0 -- a visibly self-contradictory pair. 1e-4 p.u. is far below any
# real measurement resolution.
V_TOL = 1e-4


def hourly_record():
    return {"hour": [], "vmean": [], "vbus_min": [], "vbus_max": [],
            "vph_min": [], "vph_max": [],
            "int_viol": [], "int_lo": [], "int_hi": [],
            "disch": [], "soc": [], "n_ev": [], "rho": [],
            "throughput": [], "taps": []}


def log_hour(rec, hour, feeder, disch, soc, n_ev, rho, throughput, taps=None):
    vbus = feeder.bus_vpu()
    vph = feeder.phase_vpu()
    rec["hour"].append(hour)
    rec["vmean"].append(float(np.mean(vbus)))
    rec["vbus_min"].append(float(vbus.min()))
    rec["vbus_max"].append(float(vbus.max()))
    rec["vph_min"].append(float(vph.min()))
    rec["vph_max"].append(float(vph.max()))
    # Integrated magnitude is TWO-SIDED: over- and undervoltage both count. A one-sided
    # version scores an agent that pushes the feeder above 1.05 as violation-free.
    lo = float(np.clip(V_MIN - V_TOL - vph, 0, None).sum())
    hi = float(np.clip(vph - V_MAX - V_TOL, 0, None).sum())
    rec["int_lo"].append(lo)
    rec["int_hi"].append(hi)
    rec["int_viol"].append(lo + hi)
    rec["disch"].append(float(disch))
    rec["soc"].append(float(soc))
    rec["n_ev"].append(float(n_ev))
    rec["rho"].append(float(rho))
    rec["throughput"].append(float(throughput))
    rec["taps"].append(list(taps) if taps else [])


def rainflow_depths(soc_series):
    """Compact rainflow: turning-point extraction then range counting.

    Returns the list of half-cycle depths (in SOC fraction). Used for evaluation
    only -- it is path-dependent and awkward inside an RL reward, so the reward
    uses Ah-throughput instead.
    """
    s = np.asarray(soc_series, dtype=float)
    if s.size < 3:
        return []
    # keep local extrema
    tp = [s[0]]
    for i in range(1, s.size - 1):
        if (s[i] - s[i - 1]) * (s[i + 1] - s[i]) < 0:
            tp.append(s[i])
    tp.append(s[-1])
    depths, stack = [], []
    for v in tp:
        stack.append(v)
        while len(stack) >= 3:
            a, b, c = stack[-3], stack[-2], stack[-1]
            if abs(b - a) <= abs(c - b):
                depths.append(abs(b - a))
                stack.pop(-2)
            else:
                break
    for i in range(len(stack) - 1):
        depths.append(abs(stack[i + 1] - stack[i]))
    return depths


def summarize(rec, soc_series=None):
    vm = np.asarray(rec["vmean"])
    vb, vbx = np.asarray(rec["vbus_min"]), np.asarray(rec["vbus_max"])
    vp, vpx = np.asarray(rec["vph_min"]), np.asarray(rec["vph_max"])
    depths = rainflow_depths(soc_series) if soc_series is not None else []
    n_taps = 0
    prev = None
    for t in rec["taps"]:
        if prev is not None and t and list(t) != list(prev):
            n_taps += sum(1 for x, y in zip(t, prev) if x != y)
        prev = t
    return dict(
        VMean=round(float(vm.mean()), 3),
        VMin=round(float(vm.min()), 3),
        VMax=round(float(vm.max()), 3),
        # two-sided: an hour counts if anything is outside [V_MIN, V_MAX]
        ViolMean=int(((vm < V_MIN - V_TOL) | (vm > V_MAX + V_TOL)).sum()),
        ViolBus=int(((vb < V_MIN - V_TOL) | (vbx > V_MAX + V_TOL)).sum()),
        ViolPh=int(((vp < V_MIN - V_TOL) | (vpx > V_MAX + V_TOL)).sum()),
        ViolHi=int((vpx > V_MAX + V_TOL).sum()),
        VphMin=round(float(vp.min()), 3),
        VphMax=round(float(vpx.max()), 3),
        IntViol=round(float(np.sum(rec["int_viol"])), 2),
        IntLo=round(float(np.sum(rec["int_lo"])), 2),
        IntHi=round(float(np.sum(rec["int_hi"])), 2),
        Energy=round(float(np.sum(rec["disch"])), 1),
        Thru=round(float(rec["throughput"][-1]) if rec["throughput"] else 0.0, 1),
        SOCend=round(float(rec["soc"][-1]), 3),
        MaxDoD=round(float(max(depths)) if depths else 0.0, 3),
        SumDoD=round(float(sum(depths)) if depths else 0.0, 3),
        TapOps=int(n_taps),
    )


# ---- multi-seed aggregation with paired (common-random-number) comparisons ---- #
def aggregate(rows, keys=None):
    """rows: list of summarize() dicts across seeds. Returns mean/std/CI per key."""
    keys = keys or [k for k in rows[0] if isinstance(rows[0][k], (int, float))]
    out = {}
    n = len(rows)
    for k in keys:
        v = np.array([r[k] for r in rows], dtype=float)
        sd = float(v.std(ddof=1)) if n > 1 else 0.0
        out[k] = dict(mean=float(v.mean()), std=sd,
                      ci95=1.96 * sd / np.sqrt(n) if n > 1 else 0.0)
    return out


def paired_delta(rows_a, rows_b, key):
    """Paired difference a-b on identical scenarios (CRN). Returns mean, ci95, n_wins."""
    a = np.array([r[key] for r in rows_a], dtype=float)
    b = np.array([r[key] for r in rows_b], dtype=float)
    d = a - b
    n = d.size
    sd = float(d.std(ddof=1)) if n > 1 else 0.0
    return dict(mean=float(d.mean()), ci95=1.96 * sd / np.sqrt(n) if n > 1 else 0.0,
                std=sd, n=n, a_better=int((d < 0).sum()), ties=int((d == 0).sum()))


def fmt_table(title, header, rows):
    w = [max(len(str(header[i])), *(len(str(r[i])) for r in rows)) + 2
         for i in range(len(header))]
    line = "".join(str(header[i]).rjust(w[i]) for i in range(len(header)))
    print(f"\n{title}")
    print(line)
    print("-" * len(line))
    for r in rows:
        print("".join(str(r[i]).rjust(w[i]) for i in range(len(r))))


In [ ]:
%%writefile v2g_env2.py
"""
Closed-loop training environment for the V2G gap study.

What this changes relative to the paper's two-phase setup, and why:

  * Episode = ONE DAY (hours 06:00-23:00) driven by the daily load shape, with the
    peak multiplier sampled per episode. The paper samples the load multiplier
    i.i.d. per step from [0.1, 4.0], which leaves the MDP with no temporal
    structure -- intertemporal rationing is not representable there.
  * The EV fleet is IN THE TRAINING LOOP, so SOC depletes as the agent discharges.
    The paper applies the fleet only at deployment (Phase 2) via rho-clipping.
  * SOC, availability and hour-of-day are IN THE STATE, so the agent can condition
    on how much energy it has left and how much of the day remains.
  * The reward carries an explicit Ah-throughput (degradation) term. The paper's
    reward is purely voltage (Eqs. 8-10) and has no battery term at all.

Action modes:
  "direct"    p = a * P_rated                       (the paper's formulation, Eq. 7)
  "residual"  p = clip(droop_p + a * P_rated, ...)  (droop prior; a=0 reproduces droop
                                                     at each STEP -- note this is a
                                                     per-step floor, NOT a day-level
                                                     performance guarantee, since
                                                     discharging early leaves less later)

Reward:
    r = R_vb - R_vp - w_deg * (battery kWh this step) / (P_rated_total * dt)

w_deg is expressed on the same scale as the voltage penalty (~100 per p.u. of
deviation), so sweeping it from 0 upward traces the violation/wear trade-off
directly. Sweep it rather than guessing one value.
"""
import numpy as np
import gymnasium as gym
from gymnasium import spaces

from v2g_sys import (CFG, Feeder, EVFleet, droop_pq, lam_profile,
                     reward_from_v, draw_availability)


class V2GDayEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, hub_buses, peak_range=(1.2, 3.3), mode="residual",
                 w_deg=0.0, control_mode="OFF", ev_in_loop=True,
                 reward_on="bus", iid_lambda=False, seed=0, cfg=CFG):
        super().__init__()
        self.cfg = cfg
        self.mode = mode
        self.w_deg = float(w_deg)
        self.peak_range = peak_range
        self.ev_in_loop = ev_in_loop
        self.reward_on = reward_on
        # iid_lambda=True reproduces the paper's Phase-1 training distribution: the load
        # multiplier is drawn i.i.d. per STEP from [0.1, 4.0], so the episode carries no
        # temporal structure. Training-only -- evaluation always uses the daily profile.
        self.iid_lambda = iid_lambda
        self.lam_iid_range = (0.1, 4.0)
        self.hours = cfg["active_hours"]
        self.rng = np.random.default_rng(seed)

        self.fd = Feeder(hub_buses, control_mode=control_mode)
        self.hubs = self.fd.hub_buses
        self.nh = len(self.hubs)
        self.fleets = {b: EVFleet(cfg) for b in self.hubs}

        n_obs = len(self.fd.buses) + 2 + 2 * self.nh
        self.observation_space = spaces.Box(-np.inf, np.inf, (n_obs,), np.float32)
        self.action_space = spaces.Box(-1.0, 1.0, (2 * self.nh,), np.float32)

        self.p_total = self.nh * cfg["P_rated"]
        self._t = 0
        self.peak = peak_range[0]

    # ------------------------------------------------------------------ #
    def _obs(self, vbus, lam):
        lam_norm = (lam - 0.1) / (4.0 - 0.1)
        hour_norm = self._t / max(1, len(self.hours) - 1)
        socs = [self.fleets[b].soc for b in self.hubs]
        navs = [self.fleets[b].n_avail(self.hours[min(self._t, len(self.hours) - 1)])
                / self.cfg["n_ev"] for b in self.hubs]
        return np.concatenate([vbus, [lam_norm, hour_norm], socs, navs]).astype(np.float32)

    def _lam_at(self, h):
        """Load multiplier for hour h. i.i.d. draw in paper-Phase-1 mode, else the profile."""
        if self.iid_lambda:
            return float(self.rng.uniform(*self.lam_iid_range))
        return float(lam_profile(self.peak)[h])

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        options = options or {}
        self.peak = float(options.get("peak",
                          self.rng.uniform(*self.peak_range)))
        avail_day = options.get("avail_day")
        soc0 = options.get("soc_init")
        for b in self.hubs:
            day = avail_day if avail_day is not None else draw_availability(
                self.rng, self.cfg["n_ev"])
            self.fleets[b].reset(n_avail_day=day, soc_init=soc0)
        self._t = 0
        lam0 = self._lam_at(self.hours[0])
        self.fd.set_load(lam0); self.fd.zero_hubs(); self.fd.solve()
        return self._obs(self.fd.bus_vpu(), lam0), {}

    # ------------------------------------------------------------------ #
    def _setpoint(self, b, i, a):
        P, Q = self.cfg["P_rated"], self.cfg["Q_rated"]
        if self.mode == "residual":
            v = self.fd.hub_vpu(b)
            p_d, q_d = droop_pq(v, P, Q)
            p = float(np.clip(p_d + a[2 * i] * P, -P, P))
            q = float(np.clip(q_d + a[2 * i + 1] * Q, -Q, Q))
        else:                                   # "direct" -- the paper's Eq. 7
            p = float(np.clip(a[2 * i] * P, -P, P))
            q = float(np.clip(a[2 * i + 1] * Q, -Q, Q))
        return p, q

    def step(self, action):
        h = self.hours[self._t]
        self.fd.set_load(self._lam_at(h)); self.fd.zero_hubs(); self.fd.solve()

        thru_before = sum(self.fleets[b].throughput for b in self.hubs)
        p_sup_total = 0.0
        p_batt_uncapped = 0.0
        for i, b in enumerate(self.hubs):
            p, q = self._setpoint(b, i, np.asarray(action, dtype=float))
            if self.ev_in_loop:
                p, q, _, _ = self.fleets[b].apply(p, q, h, commit=True)
            else:
                # fleet model disabled: no SOC/availability limit, but still account the
                # battery energy the command implies, so energy columns stay comparable.
                p_batt_uncapped += abs(p) / self.cfg["eta_inv"]
            self.fd.set_hub(b, p, q)
            p_sup_total += max(0.0, p)
        self.fd.solve()
        thru = (sum(self.fleets[b].throughput for b in self.hubs) - thru_before
                if self.ev_in_loop else p_batt_uncapped)

        v = self.fd.phase_vpu() if self.reward_on == "phase" else self.fd.bus_vpu()
        r = reward_from_v(v, self.cfg["v_min"], self.cfg["v_max"])
        r -= self.w_deg * (thru / max(1e-6, self.p_total))

        self._t += 1
        done = self._t >= len(self.hours)
        nh = self.hours[min(self._t, len(self.hours) - 1)]
        obs = self._obs(self.fd.bus_vpu(), self._lam_at(nh))
        return obs, float(r), bool(done), False, {"thru": thru, "p_sup": p_sup_total}


In [ ]:
%%writefile v2g_study.py
"""
Study driver: paired scenarios, controller rollouts, and the five experiments.

Two structural rules enforced here:

1. ONE LIVE CIRCUIT. Feeder.__init__ issues Clear/Compile, which resets the global
   OpenDSS state and silently invalidates any previously built Feeder. So each
   experiment builds exactly one V2GDayEnv, trains on it, and evaluates every
   controller through env.fd / env.fleets. Never hold two feeders at once.

2. COMMON RANDOM NUMBERS. Every controller in a comparison is evaluated on the
   IDENTICAL scenario list -- same availability realisation, same initial SOC, same
   peak. Differences are then paired, which removes scenario noise from the
   comparison instead of leaving it in the error bars.
"""
import time
from collections import namedtuple

import numpy as np
import torch
torch.set_num_threads(4)

from v2g_sys import CFG, droop_pq, lam_profile, draw_availability
from v2g_env2 import V2GDayEnv
import v2g_metrics as M
from stable_baselines3 import SAC

Scenario = namedtuple("Scenario", "peak avail_day soc_init")


def make_scenarios(n, peak, seed0=0, n_ev=None, soc_init=None):
    """n paired scenarios at a fixed load peak."""
    n_ev = n_ev or CFG["n_ev"]
    out = []
    for k in range(n):
        rng = np.random.default_rng(1000 + seed0 + k)
        out.append(Scenario(peak=peak,
                            avail_day=draw_availability(rng, n_ev),
                            soc_init=soc_init if soc_init is not None else CFG["soc_init"]))
    return out


# --------------------------------------------------------------------------- #
# Controller rollouts -- all driven through the env's single live feeder
# --------------------------------------------------------------------------- #
def _reset_fleets(env, scen):
    for b in env.hubs:
        env.fleets[b].reset(n_avail_day=scen.avail_day, soc_init=scen.soc_init)


def _log(env, rec, h, disch, rho, n, thru_cum):
    """thru_cum is passed explicitly: when the fleet model is disabled the fleet objects
    never accumulate, so reading fleet.throughput would silently report zero energy."""
    soc = float(np.mean([env.fleets[b].soc for b in env.hubs]))
    M.log_hour(rec, h, env.fd, disch, soc, n, rho, thru_cum, env.fd.tap_positions())


def rollout_static(env, scen, controller="droop", ev_constrained=True, damp=0.6, iters=25):
    """controller in {'baseline','droop'}. Returns summarize() dict."""
    cfg = env.cfg
    lam = lam_profile(scen.peak)
    _reset_fleets(env, scen)
    rec = M.hourly_record()
    thru_cum = 0.0
    for h in env.hours:
        env.fd.set_load(lam[h]); env.fd.zero_hubs(); env.fd.solve()
        disch, rho, n = 0.0, 1.0, np.nan

        if controller == "baseline":
            env.fd.solve()

        else:  # closed-loop droop: damped fixed point, fleet previewed but not committed
            sp = {b: (0.0, 0.0) for b in env.hubs}
            for _ in range(iters):
                for b in env.hubs:
                    v = env.fd.hub_vpu(b)
                    p, q = droop_pq(v, cfg["P_rated"], cfg["Q_rated"])
                    p0, q0 = sp[b]
                    sp[b] = ((1 - damp) * p0 + damp * p, (1 - damp) * q0 + damp * q)
                for b, (p, q) in sp.items():
                    if ev_constrained:
                        p, q, _, _ = env.fleets[b].apply(p, q, h, commit=False)
                    env.fd.set_hub(b, p, q)
                env.fd.solve()
            for b, (p, q) in sp.items():
                if ev_constrained:
                    p, q, rho, n = env.fleets[b].apply(p, q, h, commit=True)
                else:
                    thru_cum += abs(p) / cfg["eta_inv"]
                env.fd.set_hub(b, p, q)
                disch += max(0.0, p)
            env.fd.solve()

        if ev_constrained:
            thru_cum = float(sum(env.fleets[b].throughput for b in env.hubs))
        _log(env, rec, h, disch, rho, n, thru_cum)

    socs = env.fleets[env.hubs[0]].soc_series
    return M.summarize(rec, socs), rec


def rollout_policy(env, policy, scen, ev_constrained=True):
    """Deterministic policy rollout through the env itself (guarantees obs consistency)."""
    saved, saved_iid = env.ev_in_loop, env.iid_lambda
    env.ev_in_loop = ev_constrained
    env.iid_lambda = False        # evaluation always uses the real daily load profile
    obs, _ = env.reset(options=dict(peak=scen.peak, avail_day=scen.avail_day,
                                    soc_init=scen.soc_init))
    rec = M.hourly_record()
    thru_cum = 0.0
    for t, h in enumerate(env.hours):
        act, _ = policy.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(act)
        thru_cum += info["thru"]
        _log(env, rec, h, info["p_sup"], 1.0, np.nan, thru_cum)
        if done:
            break
    env.ev_in_loop, env.iid_lambda = saved, saved_iid
    socs = env.fleets[env.hubs[0]].soc_series
    return M.summarize(rec, socs), rec


def rollout_zero(env, scen, ev_constrained=True):
    """a=0 in residual mode == open-loop droop; the per-step floor sanity check."""
    class _Zero:
        def predict(self, obs, deterministic=True):
            return np.zeros(env.action_space.shape, dtype=np.float32), None
    return rollout_policy(env, _Zero(), scen, ev_constrained)


# --------------------------------------------------------------------------- #
# Training
# --------------------------------------------------------------------------- #
def train_on(env, steps, seed=0, chunk=5000, label=""):
    m = SAC("MlpPolicy", env, learning_rate=3e-4, batch_size=256, gamma=0.99,
            buffer_size=200_000, learning_starts=1000, tau=0.005,
            policy_kwargs=dict(net_arch=[256, 256]), device="cpu",
            seed=seed, verbose=0)
    t0, done = time.time(), 0
    while done < steps:
        n = min(chunk, steps - done)
        m.learn(total_timesteps=n, reset_num_timesteps=(done == 0), progress_bar=False)
        done += n
        print(f"      [{label}] {done}/{steps}  {time.time()-t0:.0f}s", flush=True)
    return m


def build_env(hub_buses, mode="residual", w_deg=0.0, control_mode="OFF",
              peak_range=(1.2, 3.3), reward_on="bus", iid_lambda=False, seed=0):
    return V2GDayEnv(hub_buses, peak_range=peak_range, mode=mode, w_deg=w_deg,
                     control_mode=control_mode, reward_on=reward_on,
                     iid_lambda=iid_lambda, seed=seed)


def build_paper_env(hub_buses, control_mode="OFF", seed=0):
    """The paper's Phase-1 training setup: direct action, i.i.d. load multiplier per step,
    no fleet in the loop. Used as the reproduction baseline and as E4's comparison arm."""
    env = build_env(hub_buses, mode="direct", w_deg=0.0, control_mode=control_mode,
                    peak_range=(0.1, 4.0), iid_lambda=True, seed=seed)
    env.ev_in_loop = False
    return env


# --------------------------------------------------------------------------- #
# E0 -- fidelity calibration
# --------------------------------------------------------------------------- #
def E0_calibration(n_scen=3):
    """Which (ControlMode, droop saturation) reproduces the paper's baseline fingerprint?

    Paper Table I baseline: feeder-mean Min = 0.907 (mild) / 0.807 (aggressive),
    violation hours (mean metric) = 13 / 17.
    """
    target = {"mild": dict(VMin=0.907, ViolMean=13),
              "aggr": dict(VMin=0.807, ViolMean=17)}
    rows = []
    for cm in ["OFF", "STATIC"]:
        env = build_env(CFG["hub_bus_single"], control_mode=cm)
        for tag, peak in [("mild", CFG["peak_mild"]), ("aggr", CFG["peak_aggr"])]:
            scens = make_scenarios(n_scen, peak, seed0=0)
            rs = [rollout_static(env, s, "baseline")[0] for s in scens]
            agg = M.aggregate(rs, ["VMean", "VMin", "ViolMean", "ViolBus", "ViolPh",
                                   "IntViol", "VphMax"])
            rows.append([cm, tag,
                         f"{agg['VMean']['mean']:.3f}",
                         f"{agg['VMin']['mean']:.3f}",
                         f"{agg['ViolMean']['mean']:.1f}",
                         f"{agg['ViolBus']['mean']:.1f}",
                         f"{agg['ViolPh']['mean']:.1f}",
                         f"{agg['IntViol']['mean']:.2f}",
                         f"{target[tag]['VMin']:.3f} / {target[tag]['ViolMean']}"])
        del env
    M.fmt_table("E0  Baseline fingerprint vs paper Table I (no V2G)",
                ["ControlMode", "load", "VMean", "VMin", "ViolMean",
                 "ViolBus", "ViolPh", "IntViol", "paper VMin/Viol"], rows)
    print("\n  Pick the ControlMode whose (VMin, ViolMean) is closest to the paper column.")
    print("  ViolBus / ViolPh / IntViol show how much the mean metric hides.")
    return rows


# --------------------------------------------------------------------------- #
# E1 -- reproduction of their Tables I and II
# --------------------------------------------------------------------------- #
def E1_reproduction(control_mode="OFF", steps=20000, n_scen=5, seed=0):
    """Their table structure, with our added metric columns."""
    out = {}
    for scope, hubs in [("single", CFG["hub_bus_single"]),
                        ("multi", CFG["hub_buses_multi"])]:
        # paper-style agent: direct action, i.i.d. lambda, no fleet in training (Phase 1)
        env = build_paper_env(hubs, control_mode=control_mode, seed=seed)
        print(f"\n  [E1/{scope}] training paper-style agent (direct, i.i.d. lambda, no fleet)")
        pol = train_on(env, steps, seed=seed, label=f"E1-{scope}")
        rows = []
        for tag, peak in [("mild", CFG["peak_mild"]), ("aggr", CFG["peak_aggr"])]:
            scens = make_scenarios(n_scen, peak, seed0=0)
            cases = [
                ("Baseline",        lambda s: rollout_static(env, s, "baseline")[0]),
                ("RL (no EV)",      lambda s: rollout_policy(env, pol, s, False)[0]),
                ("Droop (no EV)",   lambda s: rollout_static(env, s, "droop", False)[0]),
                ("RL (EV-constr)",  lambda s: rollout_policy(env, pol, s, True)[0]),
                ("Droop (EV-con.)", lambda s: rollout_static(env, s, "droop", True)[0]),
            ]
            for name, fn in cases:
                rs = [fn(s) for s in scens]
                a = M.aggregate(rs)
                rows.append([f"{tag}/{name}",
                             f"{a['VMean']['mean']:.3f}", f"{a['VMin']['mean']:.3f}",
                             f"{a['VphMax']['mean']:.3f}",
                             f"{a['ViolMean']['mean']:.1f}", f"{a['ViolBus']['mean']:.1f}",
                             f"{a['ViolPh']['mean']:.1f}", f"{a['ViolHi']['mean']:.1f}",
                             f"{a['IntViol']['mean']:.2f}",
                             f"{a['Thru']['mean']:.0f}", f"{a['SOCend']['mean']:.3f}"])
                out[f"{scope}/{tag}/{name}"] = rs
        M.fmt_table(f"E1  {scope}-hub reproduction  (ControlMode={control_mode}, {n_scen} paired scenarios)",
                    ["case", "VMean", "VMin", "VphMax", "ViolMean", "ViolBus",
                     "ViolPh", "ViolHi", "IntViol", "Thru", "SOCend"], rows)
        del env, pol
    return out


# --------------------------------------------------------------------------- #
# E2 (C1) -- multi-hub WITH realistic fleet constraints
# --------------------------------------------------------------------------- #
def E2_multihub_constrained(control_mode="OFF", steps=20000, n_scen=5, seeds=(0, 1, 2)):
    """Their explicitly untested case: multi-hub coordination under 45-85% availability."""
    out = {}
    for tag, peak in [("mild", CFG["peak_mild"]), ("aggr", CFG["peak_aggr"])]:
        scens = make_scenarios(n_scen, peak, seed0=0)
        rows, store = [], {}
        # controllers that need no training
        env = build_env(CFG["hub_buses_multi"], mode="residual", w_deg=0.0,
                        control_mode=control_mode, peak_range=(peak * 0.8, peak * 1.2))
        store["Baseline"] = [rollout_static(env, s, "baseline")[0] for s in scens]
        store["Droop"] = [rollout_static(env, s, "droop", True)[0] for s in scens]
        store["Droop (unconstr)"] = [rollout_static(env, s, "droop", False)[0] for s in scens]
        store["a=0 floor"] = [rollout_zero(env, s, True)[0] for s in scens]
        # trained closed-loop agent, one per seed, evaluated on the same scenarios
        rl_rows = []
        for sd in seeds:
            print(f"\n  [E2/{tag}] training closed-loop agent seed={sd}")
            pol = train_on(env, steps, seed=sd, label=f"E2-{tag}-s{sd}")
            rl_rows.append([rollout_policy(env, pol, s, True)[0] for s in scens])
            del pol
        store["RL closed-loop"] = [r for rs in rl_rows for r in rs]
        for name, rs in store.items():
            a = M.aggregate(rs)
            rows.append([name,
                         f"{a['VMean']['mean']:.3f}", f"{a['VMin']['mean']:.3f}",
                         f"{a['VphMax']['mean']:.3f}",
                         f"{a['ViolMean']['mean']:.1f}", f"{a['ViolBus']['mean']:.1f}",
                         f"{a['ViolPh']['mean']:.1f}", f"{a['ViolHi']['mean']:.1f}",
                         f"{a['IntViol']['mean']:.2f}±{a['IntViol']['ci95']:.2f}",
                         f"{a['Thru']['mean']:.0f}", f"{a['SOCend']['mean']:.3f}"])
        M.fmt_table(f"E2  multi-hub, EV-CONSTRAINED  ({tag}, {n_scen} paired scenarios x {len(seeds)} seeds)",
                    ["controller", "VMean", "VMin", "VphMax", "ViolMean", "ViolBus",
                     "ViolPh", "ViolHi", "IntViol(±ci)", "Thru", "SOCend"], rows)
        # paired deltas vs droop, the statistically meaningful comparison
        for key in ["IntViol", "ViolBus", "Thru"]:
            d = M.paired_delta(store["RL closed-loop"][:n_scen], store["Droop"], key)
            print(f"    paired RL-Droop  {key:8s}: {d['mean']:+.2f} ± {d['ci95']:.2f} "
                  f"(RL better in {d['a_better']}/{d['n']}, ties {d['ties']})")
        out[tag] = store
        del env
    return out


# --------------------------------------------------------------------------- #
# E3 (C2) -- degradation-weight sweep -> violation/wear frontier
# --------------------------------------------------------------------------- #
def E3_degradation_frontier(peak, weights=(0.0, 1.0, 3.0, 10.0, 30.0, 100.0),
                            control_mode="OFF", steps=20000, n_scen=5, seed=0,
                            hubs=None):
    """Trace the trade-off. Droop is plotted as a POINT on this frontier, not a rival."""
    hubs = hubs or CFG["hub_buses_multi"]
    scens = make_scenarios(n_scen, peak, seed0=0)
    pts, rows = [], []

    env0 = build_env(hubs, mode="residual", w_deg=0.0, control_mode=control_mode,
                     peak_range=(peak * 0.8, peak * 1.2))
    dr = [rollout_static(env0, s, "droop", True)[0] for s in scens]
    a = M.aggregate(dr)
    rows.append(["droop (reference)", "-",
                 f"{a['IntViol']['mean']:.2f}", f"{a['IntHi']['mean']:.2f}",
                 f"{a['Thru']['mean']:.0f}",
                 f"{a['ViolBus']['mean']:.1f}", f"{a['VphMax']['mean']:.3f}",
                 f"{a['SOCend']['mean']:.3f}", f"{a['MaxDoD']['mean']:.3f}"])
    pts.append(dict(w="droop", IntViol=a["IntViol"]["mean"], Thru=a["Thru"]["mean"],
                    ViolBus=a["ViolBus"]["mean"]))
    del env0

    for w in weights:
        env = build_env(hubs, mode="residual", w_deg=w, control_mode=control_mode,
                        peak_range=(peak * 0.8, peak * 1.2))
        print(f"\n  [E3] training w_deg={w}")
        pol = train_on(env, steps, seed=seed, label=f"E3-w{w}")
        rs = [rollout_policy(env, pol, s, True)[0] for s in scens]
        a = M.aggregate(rs)
        rows.append([f"RL w_deg={w:g}", f"{w:g}",
                     f"{a['IntViol']['mean']:.2f}", f"{a['IntHi']['mean']:.2f}",
                     f"{a['Thru']['mean']:.0f}",
                     f"{a['ViolBus']['mean']:.1f}", f"{a['VphMax']['mean']:.3f}",
                     f"{a['SOCend']['mean']:.3f}", f"{a['MaxDoD']['mean']:.3f}"])
        pts.append(dict(w=w, IntViol=a["IntViol"]["mean"], Thru=a["Thru"]["mean"],
                        ViolBus=a["ViolBus"]["mean"]))
        del env, pol

    M.fmt_table(f"E3  violation / wear frontier  (peak={peak}, {n_scen} paired scenarios)",
                ["controller", "w_deg", "IntViol", "IntHi", "Thru(kWh)", "ViolBus",
                 "VphMax", "SOCend", "MaxDoD"], rows)
    print("\n  Read it as a frontier: IntViol should rise as Thru falls. Where droop sits")
    print("  relative to the RL frontier is the result -- above, on, or below it.")
    return pts


# --------------------------------------------------------------------------- #
# E4 (C3) -- the multi-hub aggressive stress case (their droop 2 vs RL 15)
# --------------------------------------------------------------------------- #
def E4_stress(control_mode="OFF", steps=20000, n_scen=5, seeds=(0, 1, 2)):
    """Does day-structured, fleet-in-loop training change the aggressive-case gap?

    Also runs the paper-style agent (i.i.d. load multiplier, no fleet in training) on
    the same scenarios, so the two training regimes are compared directly.
    """
    peak = CFG["peak_aggr"]
    scens = make_scenarios(n_scen, peak, seed0=0)
    store, rows = {}, []

    env = build_env(CFG["hub_buses_multi"], mode="residual", w_deg=0.0,
                    control_mode=control_mode, peak_range=(peak * 0.8, peak * 1.2))
    store["Droop"] = [rollout_static(env, s, "droop", True)[0] for s in scens]
    store["Droop (unconstr)"] = [rollout_static(env, s, "droop", False)[0] for s in scens]
    for sd in seeds:
        print(f"\n  [E4] day-structured closed-loop agent seed={sd}")
        pol = train_on(env, steps, seed=sd, label=f"E4-day-s{sd}")
        store.setdefault("RL day-structured", []).extend(
            [rollout_policy(env, pol, s, True)[0] for s in scens])
        del pol
    del env

    env2 = build_paper_env(CFG["hub_buses_multi"], control_mode=control_mode)
    for sd in seeds:
        print(f"\n  [E4] paper-style agent (i.i.d. lambda, no fleet) seed={sd}")
        pol = train_on(env2, steps, seed=sd, label=f"E4-iid-s{sd}")
        store.setdefault("RL paper-style", []).extend(
            [rollout_policy(env2, pol, s, True)[0] for s in scens])
        del pol
    del env2

    for name, rs in store.items():
        a = M.aggregate(rs)
        rows.append([name,
                     f"{a['VMean']['mean']:.3f}", f"{a['VMin']['mean']:.3f}",
                     f"{a['VphMax']['mean']:.3f}",
                     f"{a['ViolMean']['mean']:.1f}", f"{a['ViolBus']['mean']:.1f}",
                     f"{a['ViolPh']['mean']:.1f}", f"{a['ViolHi']['mean']:.1f}",
                     f"{a['IntViol']['mean']:.2f}±{a['IntViol']['ci95']:.2f}",
                     f"{a['Thru']['mean']:.0f}"])
    M.fmt_table(f"E4  multi-hub AGGRESSIVE stress  ({n_scen} paired scenarios x {len(seeds)} seeds)",
                ["controller", "VMean", "VMin", "VphMax", "ViolMean", "ViolBus",
                 "ViolPh", "ViolHi", "IntViol(±ci)", "Thru"], rows)
    return store


def runtime_estimate(steps, n_trainings, sec_per_20k=240):
    mins = n_trainings * steps / 20000 * sec_per_20k / 60
    print(f"  ~{n_trainings} trainings x {steps} steps  ->  approx {mins:.0f} min of training")


In [ ]:
# 1. run parameters
QUICK = True          # <-- set False for the real run

if QUICK:
    STEPS, N_SCEN, SEEDS, WEIGHTS = 4000, 2, (0,), (0.0, 10.0, 100.0)
else:
    STEPS, N_SCEN, SEEDS, WEIGHTS = 20000, 5, (0, 1, 2), (0.0, 1.0, 3.0, 10.0, 30.0, 100.0)

CONTROL_MODE = "OFF"   # revisit after E0 tells you which one matches the paper

import importlib, json
import v2g_sys, v2g_metrics, v2g_env2, v2g_study
for m in (v2g_sys, v2g_metrics, v2g_env2, v2g_study):
    importlib.reload(m)
from v2g_sys import CFG
import v2g_study as S

# E1: single+multi | E2: 2 loadings x seeds | E3: 2 loadings x weights | E4: 2 arms x seeds
n_train = 2 + 2 * len(SEEDS) + 2 * len(WEIGHTS) + 2 * len(SEEDS)
print(f"QUICK={QUICK}  steps={STEPS}  scenarios={N_SCEN}  seeds={SEEDS}")
S.runtime_estimate(STEPS, n_train)
RESULTS = {}

## E0 — fidelity calibration

The paper never states whether the IEEE-34 regulators are active. Their reported no-V2G
baseline is the fingerprint: feeder-mean min **0.907** (mild) / **0.807** (aggressive), and
**13** / **17** violation hours on the mean metric. Whichever `ControlMode` reproduces that
is the setting to use everywhere else.

A pre-run of this cell during development gave:

| ControlMode | load | VMin | paper VMin | ViolMean | paper Viol |
|---|---|---|---|---|---|
| **OFF** | mild | **0.903** | 0.907 | 10 | 13 |
| **OFF** | aggr | **0.800** | 0.807 | 16 | 17 |
| STATIC | mild | 0.983 | 0.907 | 0 | 13 |
| STATIC | aggr | 0.831 | 0.807 | 13 | 17 |

So **`ControlMode=OFF` is the paper's setting** — worst-case mean voltage matches to within
0.004–0.007 p.u. on both loadings, while STATIC is off by 0.076 p.u. at mild. The residual
violation-hour gap (10 vs 13, 16 vs 17) is the reconstructed load shape, not the regulators.
Confirm on your machine, then leave `CONTROL_MODE = "OFF"`.

Also worth reading off that table: with regulators active (STATIC, mild) the **mean** metric
reports **0** violation hours while **14** hours have at least one bus below 0.95. That single
row is the clearest statement of what the mean-voltage metric hides.

In [ ]:
RESULTS["E0"] = S.E0_calibration(n_scen=max(2, N_SCEN // 2))

## E1 — reproduction of Tables I and II

Their five single-hub configurations and the multi-hub cases, using a paper-style agent
(direct action, i.i.d. load multiplier, **no fleet in training**) so this is a faithful
reproduction rather than our variant. Extra metric columns are additions, not changes.

In [ ]:
RESULTS["E1"] = S.E1_reproduction(control_mode=CONTROL_MODE, steps=STEPS,
                                  n_scen=N_SCEN, seed=0)

## E2 — multi-hub under realistic fleet constraints (C1)

Their multi-hub study assumes sufficient EV capacity at every hub. Here the same five hubs
run under the 45–85% availability profile with SOC/SOH dynamics, against closed-loop droop,
the unconstrained droop reference, and the `a=0` per-step floor.

Paired deltas against droop are printed below each table — that is the comparison with
statistical meaning, since the scenarios are shared.

In [ ]:
RESULTS["E2"] = S.E2_multihub_constrained(control_mode=CONTROL_MODE, steps=STEPS,
                                          n_scen=N_SCEN, seeds=SEEDS)

## E3 — degradation-aware objective (C2)

Their reward is purely voltage (Eqs. 8–10) and contains no battery term. Here an
Ah-throughput cost is added and its weight swept, tracing the violation/wear frontier.

`w_deg` is expressed on the same scale as the voltage penalty (~100 per p.u. of deviation),
so the sweep spans "voltage only" to "wear dominates". Droop appears as a **point** on this
plane, not as a rival — where it lands relative to the frontier is the result.

In [ ]:
RESULTS["E3_aggr"] = S.E3_degradation_frontier(
    peak=CFG["peak_aggr"], weights=WEIGHTS, control_mode=CONTROL_MODE,
    steps=STEPS, n_scen=N_SCEN, seed=0)

RESULTS["E3_mild"] = S.E3_degradation_frontier(
    peak=CFG["peak_mild"], weights=WEIGHTS, control_mode=CONTROL_MODE,
    steps=STEPS, n_scen=N_SCEN, seed=0)

## E4 — multi-hub aggressive stress (C3)

Their Table II reports droop at **2** violation hours and RL at **15** in this case, which
their conclusion flags. Two training regimes are compared on identical scenarios: the
day-structured, fleet-in-loop agent, and a paper-style agent (i.i.d. λ, no fleet in
training). If the gap is a training-distribution artifact, this is where it shows.

In [ ]:
RESULTS["E4"] = S.E4_stress(control_mode=CONTROL_MODE, steps=STEPS,
                            n_scen=N_SCEN, seeds=SEEDS)

## Figures

In [ ]:
import numpy as np, matplotlib.pyplot as plt

def frontier_fig(pts, tag, fname):
    rl = [p for p in pts if p["w"] != "droop"]
    dr = [p for p in pts if p["w"] == "droop"]
    fig, ax = plt.subplots(figsize=(6.4, 4.6))
    if rl:
        x = [p["Thru"] for p in rl]; y = [p["IntViol"] for p in rl]
        order = np.argsort(x)
        ax.plot(np.array(x)[order], np.array(y)[order], "o-", color="#27ae60",
                label="RL frontier (w_deg sweep)")
        for p in rl:
            ax.annotate(f"w={p['w']:g}", (p["Thru"], p["IntViol"]),
                        textcoords="offset points", xytext=(5, 5), fontsize=8)
    if dr:
        ax.plot([dr[0]["Thru"]], [dr[0]["IntViol"]], "D", ms=10, color="#c0392b",
                label="closed-loop droop")
    ax.set_xlabel("battery throughput (kWh/day)")
    ax.set_ylabel("integrated violation magnitude (p.u.·h)")
    ax.set_title(f"Violation / wear frontier — multi-hub {tag}")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(fname, dpi=120, bbox_inches="tight"); plt.show()

for tag in ["aggr", "mild"]:
    key = f"E3_{tag}"
    if key in RESULTS:
        frontier_fig(RESULTS[key], tag, f"fig_frontier_{tag}.png")

json.dump({k: v for k, v in RESULTS.items() if k.startswith("E3")},
          open("frontier_points.json", "w"), indent=1, default=str)
print("saved frontier_points.json")

## What to send back

1. **E0 table** — and which `ControlMode` matched the paper's fingerprint.
2. **E1 tables** (single and multi) — especially how far `ViolPh` sits from `ViolMean`.
3. **E2 tables + the paired-delta lines**.
4. **E3 tables** for both loadings, and `fig_frontier_aggr.png` / `fig_frontier_mild.png`.
5. **E4 table** — day-structured vs paper-style training.
6. Any cell that errored, with the traceback.

Whatever the numbers say is the result — including droop landing on or below the frontier.
That outcome is reportable and is what the paper itself found under stress.